# Global Store · 9.800 pedidos

Pedidos de una tienda global (18 columnas, 2015 en adelante) analizados con pandas, sin Fabric. El CSV `train.csv` se lee directo del repositorio.

Lo que responde: cuanto pesa cada modo de envio, como evolucionan las ventas mes a mes y que categorias mueven el negocio.

In [ ]:
import io, urllib.request
import pandas as pd
import matplotlib.pyplot as plt

URL = "https://raw.githubusercontent.com/Nicolenki7/Tech_GlobalStore_Retail_Analysis-Fabric/main/train.csv"
with urllib.request.urlopen(URL) as r:
    df = pd.read_csv(io.BytesIO(r.read()))

print("filas:", len(df), "| columnas:", len(df.columns))
print("columnas:", list(df.columns))

# las fechas del dataset estan en formato dia/mes/anio
for col in ("Order Date", "Ship Date"):
    df[col] = pd.to_datetime(df[col], format="%d/%m/%Y", errors="coerce")

print()
print("pedidos unicos:", df["Order ID"].nunique())
print("rango de fechas:", df["Order Date"].min().date(), "->", df["Order Date"].max().date())
print("fechas ilegibles:", df["Order Date"].isna().sum())
print(f"ventas totales: {df['Sales'].sum():,.2f}")

In [ ]:
por_envio = (df.groupby("Ship Mode")["Sales"]
               .agg(pedidos="count", venta="sum")
               .sort_values("venta", ascending=False))
por_envio["pct"] = (por_envio["venta"] / por_envio["venta"].sum() * 100).round(2)

ax = por_envio["venta"].plot(kind="bar", figsize=(9, 4), color="#457b9d")
ax.set_title("Ventas por modo de envio")
ax.set_ylabel("Sales")
ax.set_xlabel("")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

por_envio

In [ ]:
mes = (df.set_index("Order Date")["Sales"]
         .resample("MS").sum())

ax = mes.plot(kind="line", marker="o", figsize=(10, 4), color="#2a9d8f")
ax.set_title("Ventas por mes")
ax.set_ylabel("Sales")
ax.set_xlabel("")
plt.tight_layout()
plt.show()

print(f"meses con actividad: {(mes > 0).sum()} | mejor mes: {mes.idxmax().strftime('%Y-%m')} ({mes.max():,.2f})")
mes.tail(12)

In [ ]:
cat = (df.groupby("Category")["Sales"].sum().sort_values(ascending=False))
sub = (df.groupby("Sub-Category")["Sales"].sum().sort_values(ascending=False).head(8))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
cat.plot(kind="bar", ax=axes[0], color="#e76f51")
axes[0].set_title("Ventas por categoria")
axes[0].set_xlabel("")
sub.plot(kind="barh", ax=axes[1], color="#8ab17d")
axes[1].invert_yaxis()
axes[1].set_title("Top 8 sub-categorias")
axes[1].set_xlabel("Sales")
plt.tight_layout()
plt.show()

print("categorias:", len(cat), "| sub-categorias:", df['Sub-Category'].nunique())
print(f"la principal concentra el {cat.iloc[0] / cat.sum() * 100:.1f}% de las ventas")
sub